# Predicting Vulnerable Marine Ecosystem (VME) Habitat in Indonesian Deep Waters Based on Indicator Species Presence

VMEs are groups of species, communities or habitats that may be vulnerable to impacts from fishing activities. The vulnerability of an ecosystem is related to the vulnerability of its population, communities or habitats.

An indicator species is a species whose presence, absence, or abundance reflects specific environmental conditions or ecosystem characteristics. In marine ecology, they act as biological “signals” of certain habitat types.

When indicator species are observed in a location, their presence suggests that the underlying habitat conditions suitable for VMEs may exist there. However, biodiversity data is harder to find compared to environmental data, so by analysing the environmental data in places of indicator species sightings, VMEs can be predicted in locations with similar environmental data.

This notebook builds a species distribution model (SDM) pipeline to predict VME likelihood 
across Indonesian deep sea (depth more than 200 meters). Biodiversity data is sourced from OBIS, indicator species is identified using WoRMS and matched against environmental predictors from Bio-ORACLE and The GEBCO Grid bathymetry. 

Six machine learning models are trained, evaluated, and combined into an ensemble prediction map to predict potential VMEs in Indonesia region. 

In [27]:
import requests, zipfile, io, pandas as pd
import numpy as np
import xarray as xr
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from sklearn.metrics import classification_report, accuracy_score, recall_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from pygam import LogisticGAM
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import elapid
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Data Collection

These data sources are combined to build the modelling dataset:

- **OBIS (Ocean Biodiversity Information System)**: provides occurrence records of benthic species     in Indonesia, filtered to depths >200 m
- **The GEBCO Grid (General Bathymetric Chart of the Oceans**: provides high-resolution bathymetry data
- **WoRMS (World Register of Marine Species)**: provides taxonomic data for species in occurence        records to identify indicator species. 
- **Bio-ORACLE**: provides environmental data at benthic layer

The study area covers Indonesian waters bounded by 100°E–145°E longitude and 12°S–7°N latitude.

In [28]:
#biodiversity dataset used is 'Benthic species from the tropical Pacific surrounding New Caledonia' from OBIS
#online dataset: https://obis.org/dataset/e971710e-054e-43f2-b23c-c149d6c76cb4
url = "https://ipt.vliz.be/eurobis/archive.do?r=new_caledonia"
response = requests.get(url)

#extract zip file and read occurrence file
z = zipfile.ZipFile(io.BytesIO(response.content))
df = pd.read_csv(z.open('occurrence.txt'),index_col=0 , sep='\t')

print("Biodiversity dataset downloaded!")

Biodiversity dataset downloaded!


In [29]:
#clean dataset
#drop columns where all values are missing
cleaned_df = df.dropna(axis=1,how='all')

#drop unnecessary columns
cols = ["decimalLatitude","decimalLongitude","minimumDepthInMeters","maximumDepthInMeters","scientificName"]
cleaned_df = cleaned_df[cols]

#check info of cleaned dataset
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58156 entries, urn:catalog:IRD-MNHN:NewCaledonia::0x0073A50101000100 to urn:catalog:IRD-MNHN:NewCaledonia:9999:0x5DA6A50101000E00
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   decimalLatitude       58156 non-null  float64
 1   decimalLongitude      58156 non-null  float64
 2   minimumDepthInMeters  56858 non-null  float64
 3   maximumDepthInMeters  56810 non-null  float64
 4   scientificName        58156 non-null  object 
dtypes: float64(4), object(1)
memory usage: 2.7+ MB


In [ ]:
# depth values from OBIS unreliable
# replace OBIS depth with seafloor depth from The GEBCO Grid; assume all benthic organisms are found on benthic level

depth_ds = xr.open_dataset("gebco_2025_n7.0_s-12.0_w100.0_e145.0.nc",engine="netcdf4")

#attach seafloor depth value from The GEBCO Grid to each point in biodiversity dataset
cleaned_df["depth"] = depth_ds["elevation"].sel(
    lat=xr.DataArray(cleaned_df["decimalLatitude"], dims="points"),
    lon=xr.DataArray(cleaned_df["decimalLongitude"], dims="points"),
    method="nearest"
).values

#drop depth values from OBIS
cleaned_df = cleaned_df.drop(columns=["minimumDepthInMeters", "maximumDepthInMeters"])

In [31]:
#set biodiversity dataset to only deep-sea observations (depth > 200 m)
deep_df=cleaned_df[cleaned_df["depth"]<=-200]

#set biodiversity dataset to only Indonesia region (lon: 100–145, lat: -12–7)
indo_df = deep_df[
    (deep_df["decimalLongitude"] >= 100) &
    (deep_df["decimalLongitude"] <= 145) &
    (deep_df["decimalLatitude"] >= -12) &
    (deep_df["decimalLatitude"] <= 7)
]

indo_df = indo_df.copy()

In [32]:
#get genus from scientific name in indo_df
indo_df["genus"] = indo_df["scientificName"].str.split().str[0]

#function to get classification from WoRMS API
def get_worms_classification(genus):
    url = f"https://www.marinespecies.org/rest/AphiaRecordsByName/{genus}?like=false&marine_only=true"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200 or len(r.json()) == 0:
            return None
        data = r.json()[0]
        return {"phylum": data.get("phylum"), "order": data.get("order"), "family": data.get("family")}
    except:
        return None

#classify genus as indicator species of VME (1) or not (0) based on taxonomy
#taxonomy based on: https://www.fao.org/in-action/vulnerable-marine-ecosystems/vme-indicators/en
def classify_genus(genus):
    tax = get_worms_classification(genus)
    if tax is None:
        return 0
    if tax["phylum"] == "Porifera":
        return 1
    if tax["order"] in ["Scleractinia", "Antipatharia", "Alcyonacea", "Pennatulacea", "Actiniaria"]:
        return 1
    if tax["family"] == "Stylasteridae":
        return 1
    return 0

#get all unique genus to avoid redundancy in API calls
unique_genera = indo_df["genus"].dropna().unique()
genus_to_vme = {}

#run classification in parallel (faster than looping one by one)
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(classify_genus, g): g for g in unique_genera}
    for i, future in enumerate(as_completed(futures)):
        g = futures[future]
        genus_to_vme[g] = future.result()
        print(f"{i+1}/{len(unique_genera)} — {g}", end="\r")

#map VME labels back to indo_df
indo_df["VME"] = indo_df["genus"].map(genus_to_vme)
print("\nVME indicator done!")
indo_df["VME"].value_counts()

359/359 — Calliotropisolaesm
VME indicator done!


VME
0    1770
1     583
Name: count, dtype: int64

In [33]:
#download all environmental layers (lat and lon of Indonesia region already in URL)
layers = {
    "temp_mean":        "https://erddap.bio-oracle.org/erddap/griddap/thetao_baseline_2000_2019_depthmean.nc?thetao_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "temp_ltmax":       "https://erddap.bio-oracle.org/erddap/griddap/thetao_baseline_2000_2019_depthmean.nc?thetao_ltmax%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "temp_ltmin":       "https://erddap.bio-oracle.org/erddap/griddap/thetao_baseline_2000_2019_depthmean.nc?thetao_ltmin%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "salinity_mean":    "https://erddap.bio-oracle.org/erddap/griddap/so_baseline_2000_2019_depthmean.nc?so_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "nitrate_mean":     "https://erddap.bio-oracle.org/erddap/griddap/no3_baseline_2000_2018_depthmean.nc?no3_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "phosphate_mean":   "https://erddap.bio-oracle.org/erddap/griddap/po4_baseline_2000_2018_depthmean.nc?po4_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "oxygen_mean":      "https://erddap.bio-oracle.org/erddap/griddap/o2_baseline_2000_2018_depthmean.nc?o2_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "productivity_mean":"https://erddap.bio-oracle.org/erddap/griddap/phyc_baseline_2000_2020_depthmean.nc?phyc_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "ph_mean":          "https://erddap.bio-oracle.org/erddap/griddap/ph_baseline_2000_2018_depthmean.nc?ph_mean%5B(2010-01-01):1:(2010-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "slope":            "https://erddap.bio-oracle.org/erddap/griddap/terrain_characteristics.nc?slope%5B(1970-01-01T00:00:00Z):1:(1970-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "tpi":              "https://erddap.bio-oracle.org/erddap/griddap/terrain_characteristics.nc?topographic_position_index%5B(1970-01-01T00:00:00Z):1:(1970-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
    "tri":              "https://erddap.bio-oracle.org/erddap/griddap/terrain_characteristics.nc?terrain_ruggedness_index%5B(1970-01-01T00:00:00Z):1:(1970-01-01T00:00:00Z)%5D%5B(-12.0):1:(7.0)%5D%5B(100.0):1:(145.0)%5D",
}

#loop through each layer and save as .nc file
for name, url in layers.items():
    print(f"Downloading {name}...")
    urllib.request.urlretrieve(url, f"{name}.nc")
    print(f"Done: {name}.nc")

print("All downloads complete!")

# map each file to its variable name inside indo_df
varnames = {
    "temp_mean":         "thetao_mean",
    "temp_ltmax":        "thetao_ltmax",
    "temp_ltmin":        "thetao_ltmin",
    "salinity_mean":     "so_mean",
    "nitrate_mean":      "no3_mean",
    "phosphate_mean":    "po4_mean",
    "oxygen_mean":       "o2_mean",
    "productivity_mean": "phyc_mean",
    "ph_mean":           "ph_mean",
    "slope":             "slope",
    "tpi":               "topographic_position_index",
    "tri":               "terrain_ruggedness_index",
}

lons = xr.DataArray(indo_df["decimalLongitude"].values, dims="points")
lats = xr.DataArray(indo_df["decimalLatitude"].values, dims="points")

for filename, varname in varnames.items():
    env_ds= xr.open_dataset(f"{filename}.nc")
    values = env_ds[varname].sel(
        longitude=lons,
        latitude=lats,
        method="nearest"
    ).values
    indo_df[filename] = values.squeeze()
    env_ds.close()
    print(f"Added {filename}")

print("Environmental data downloaded!")
print(indo_df.shape)

Done: temp_mean.nc
Done: temp_ltmax.nc
Done: temp_ltmin.nc
Done: salinity_mean.nc
Done: nitrate_mean.nc
Done: phosphate_mean.nc
Done: oxygen_mean.nc
Done: productivity_mean.nc
Done: ph_mean.nc
Done: slope.nc
Done: tpi.nc
Done: tri.nc
All downloads complete!
Added temp_mean
Added temp_ltmax
Added temp_ltmin
Added salinity_mean
Added nitrate_mean
Added phosphate_mean
Added oxygen_mean
Added productivity_mean
Added ph_mean
Added slope
Added tpi
Added tri
Environmental data downloaded!
(2353, 18)


In [34]:
#keep only rows with complete environmental + depth data
features=['depth', 'temp_mean', 'temp_ltmax', 'temp_ltmin', 'salinity_mean', 'nitrate_mean','phosphate_mean','oxygen_mean','productivity_mean','ph_mean','slope','tpi','tri']
indo_df = indo_df.dropna(subset=features)

print(indo_df.shape)  # should be 2348

(2348, 18)


## Species Distribution Modelling

Six model types are trained and compared with each other:

- **GLM**
- **GAM**
- **Decision Tree**
- **Random Forest**
- **Gradient Boosting (XGBoost)**
- **MaxEnt**

Final predictions are averaged into an ensemble, which is more stable than relying on any single model. The ensemble probability score at each grid point is mapped across target area to visualise VME habitat suitability.

In [35]:
#try different models: GLM, GAM, decision tree, random forest, boosted trees, maximum entropy for VME prediction
#use GridSearchCV for most models to find best parameters for model
#use manual search for maximum entropy to find best parameters for model

#X = input features, y = target (VME classification)
X = indo_df[features]
y = indo_df['VME']

#first divide indo_df to 70% training data & 30% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,shuffle=True,stratify=y,random_state=42)

In [36]:
#LogisticRegression
#needs to use scaled x data

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

glm_search = GridSearchCV(LogisticRegression(class_weight='balanced', random_state=42), param_grid={'max_iter': [500, 1000, 2000]}, cv=10, scoring='recall')

glm_search.fit(X_train_scaled, y_train)
best_glm = glm_search.best_estimator_
print('Best GLM params:', glm_search.best_params_)

y_pred_glm = best_glm.predict(X_test_scaled)
accuracy_glm = accuracy_score(y_test, y_pred_glm)
recall_glm = recall_score(y_test, y_pred_glm)

Best GLM params: {'max_iter': 500}


In [37]:
#LogisticGAM
#needs to use scaled x data

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gam_search = GridSearchCV(LogisticGAM(),param_grid={'max_iter': [500, 1000, 2000], 'lam': [0.1, 0.6, 1.0]},cv=10, scoring='recall')

gam_search.fit(X_train_scaled, y_train,weights=sample_weights)
best_gam = gam_search.best_estimator_
print('Best GAM params:', gam_search.best_params_)

y_pred_gam = best_gam.predict(X_test_scaled)
accuracy_gam = accuracy_score(y_test, y_pred_gam)
recall_gam = recall_score(y_test, y_pred_gam)

c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\links.py:137: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\pygam.py:631: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\links.py:121: RuntimeWarning: overflow encountered in exp
  elp = np.exp(lp)
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\links.py:122: RuntimeWarning: invalid value encountered in divide
  return dist.levels * elp / (elp + 1)
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\pygam.py:631: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\links.py:137: RuntimeWarning: overflow encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
c:\Users\Hanna\anaconda3\Lib\site-packages\pygam\links.py:137: RuntimeW

Best GAM params: {'lam': 0.1, 'max_iter': 500}


In [38]:
#DecisionTreeClassifier

dt_search = GridSearchCV(DecisionTreeClassifier(class_weight='balanced',random_state=42),param_grid={'max_depth':[None,5,6,7,8,9,10]},cv=10, scoring='recall')

dt_search.fit(X_train, y_train)
best_dt = dt_search.best_estimator_
print('Best Decision Tree params:', dt_search.best_params_)

y_pred_dt = best_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)

Best Decision Tree params: {'max_depth': 9}


In [39]:
#RandomForestClassifier

rf_search = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),param_grid={'n_estimators':[100, 200, 300], 'max_depth':[None, 5, 10, 15, 20, 25]},cv=10, scoring='recall')

rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_
print('Best Random Forest params:', rf_search.best_params_)

y_pred_rf = best_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)

Best Random Forest params: {'max_depth': None, 'n_estimators': 300}


In [40]:
#XGBClassifier #BoostedTrees

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

gb_search = GridSearchCV(XGBClassifier(scale_pos_weight=neg/pos, random_state=42, eval_metric='logloss'), param_grid={'n_estimators': [200, 300, 400, 500], 'max_depth': [5, 10, 15, 20, 25]}, cv=10, scoring='recall')
gb_search.fit(X_train, y_train)
best_gb = gb_search.best_estimator_
print('Best Grad Boost params:', gb_search.best_params_)

y_pred_gb = best_gb.predict(X_test)
accuracy_gb = accuracy_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)

Best Grad Boost params: {'max_depth': 20, 'n_estimators': 400}


In [41]:
#Maximum Entropy

betas      = [0.5, 1.0, 1.5, 2.0, 3.0]
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
min_accuracy = 0.6

best_recall_maxent = 0
best_model_maxent  = None
best_threshold     = None

for b in betas:
    m = elapid.MaxentModel(beta_multiplier=b)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    for t in thresholds:
        y_bin = (y_pred >= t).astype(int)
        rec = recall_score(y_test, y_bin)
        acc = accuracy_score(y_test, y_bin)
        if rec > best_recall_maxent and acc >= min_accuracy:
            best_recall_maxent = rec
            maxent_best_combo  = {'beta': b, 'threshold': t}
            best_model_maxent  = m
            best_threshold     = t

y_pred_maxent_bin = (best_model_maxent.predict(X_test) >= best_threshold).astype(int)
accuracy_maxent   = accuracy_score(y_test, y_pred_maxent_bin)
recall_maxent     = recall_score(y_test, y_pred_maxent_bin)

In [42]:
#results of each model with best parameters
results = []
results.append({'Model': 'GLM (best)',           'Best Params': glm_search.best_params_, 'Accuracy': accuracy_glm, 'VME Recall': recall_glm})
results.append({'Model': 'GAM (best)',           'Best Params': gam_search.best_params_, 'Accuracy': accuracy_gam, 'VME Recall': recall_gam})
results.append({'Model': 'Decision Tree (best)', 'Best Params': dt_search.best_params_,  'Accuracy': accuracy_dt,  'VME Recall': recall_dt})
results.append({'Model': 'Random Forest (best)', 'Best Params': rf_search.best_params_,  'Accuracy': accuracy_rf,  'VME Recall': recall_rf})
results.append({'Model': 'GradBoost (best)',     'Best Params': gb_search.best_params_,  'Accuracy': accuracy_gb,  'VME Recall': recall_gb})
results.append({'Model': 'MaxEnt (best)',        'Best Params': maxent_best_combo,       'Accuracy': accuracy_maxent, 'VME Recall': recall_maxent})

results_df = pd.DataFrame(results)
results_df

,Model,Best Params,Accuracy,VME Recall
0,GLM (best),{'max_iter': 500},0.602837,0.634286
1,GAM (best),"{'lam': 0.1, 'max_iter': 500}",0.675177,0.742857
2,Decision Tree (best),{'max_depth': 9},0.693617,0.805714
3,Random Forest (best),"{'max_depth': None, 'n_estimators': 300}",0.751773,0.800000
4,GradBoost (best),"{'max_depth': 20, 'n_estimators': 400}",0.729078,0.834286
5,MaxEnt (best),"{'beta': 2.0, 'threshold': 0.4}",0.611348,0.800000


In [43]:
probas = {
    'GLM':           best_glm.predict_proba(X_test_scaled)[:, 1],
    'GAM':           best_gam.predict_proba(X_test_scaled),
    'Decision Tree': best_dt.predict_proba(X_test)[:, 1],
    'Random Forest': best_rf.predict_proba(X_test)[:, 1],
    'GradBoost':     best_gb.predict_proba(X_test)[:, 1],
    'MaxEnt':        best_model_maxent.predict(X_test),
}
model_names   = ['GLM', 'GAM', 'Decision Tree', 'Random Forest', 'GradBoost', 'MaxEnt']
std_per_model = np.array([probas[m].std() for m in model_names])


In [ ]:
#build prediction grid over Indonesia region
#interval = grid resolution (0.1° ~11 km, smaller = finer but slower)
 
interval = 0.1
latitudegrid = np.arange(-12.0, 7.0, interval)
longitudegrid = np.arange(100.0, 145.0, interval)
lon_grid, lat_grid = np.meshgrid(longitudegrid, latitudegrid)
 
grid_df = pd.DataFrame({
    "Latitude": lat_grid.ravel(),
    "Longitude": lon_grid.ravel(),
})

#attach seafloor depth values from The GEBCO Grid to each point in prediction grid
grid_df["depth"] = depth_ds["elevation"].sel(
    lat = xr.DataArray(grid_df["Latitude"], dims="points"),
    lon = xr.DataArray(grid_df["Longitude"], dims="points"),
    method="nearest"
).values

#set prediction grid to only deep-sea observations (depth > 200 m)
grid_df = grid_df[grid_df["depth"]<=-200].reset_index(drop=True)

lons = xr.DataArray(grid_df["Longitude"].values, dims="points")
lats = xr.DataArray(grid_df["Latitude"].values, dims="points")

# extract environmental variables to each point in prediction grid
for filename, varname in varnames.items():
    env_ds= xr.open_dataset(f"{filename}.nc")
    values = env_ds[varname].sel(
        longitude=lons,
        latitude=lats,
        method="nearest"
    ).values
    grid_df[filename] = values.squeeze()
    env_ds.close()
    print(f"Added {filename}")

#keep only rows with complete environmental + depth data
grid_df=grid_df.dropna(subset=features)
print("Data for VME prediction map downloaded!")
grid_df.info()

Added temp_mean
Added temp_ltmax
Added temp_ltmin
Added salinity_mean
Added nitrate_mean
Added phosphate_mean
Added oxygen_mean
Added productivity_mean
Added ph_mean
Added slope
Added tpi
Added tri
Data for VME prediction map downloaded!
<class 'pandas.core.frame.DataFrame'>
Index: 43098 entries, 0 to 43131
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Latitude           43098 non-null  float64
 1   Longitude          43098 non-null  float64
 2   depth              43098 non-null  float32
 3   temp_mean          43098 non-null  float64
 4   temp_ltmax         43098 non-null  float64
 5   temp_ltmin         43098 non-null  float64
 6   salinity_mean      43098 non-null  float64
 7   nitrate_mean       43098 non-null  float64
 8   phosphate_mean     43098 non-null  float64
 9   oxygen_mean        43098 non-null  float64
 10  productivity_mean  43098 non-null  float64
 11  ph_mean            43098 non-

In [45]:
#run prediction models on prediction map dataset 
X_pred = grid_df[features]
X_pred_scaled = scaler.transform(X_pred)  # for models that need scaling (GLM, GAM)
 
grid_df["prob_glm"]    = best_glm.predict_proba(X_pred_scaled)[:, 1]
grid_df["prob_gam"]    = best_gam.predict_proba(X_pred_scaled)
grid_df["prob_dt"]     = best_dt.predict_proba(X_pred)[:, 1]
grid_df["prob_rf"]     = best_rf.predict_proba(X_pred)[:, 1]
grid_df["prob_gb"]     = best_gb.predict_proba(X_pred)[:, 1]
grid_df["prob_maxent"] = best_model_maxent.predict(X_pred)
# MaxEnt — already outputs 0-1 suitability score

#combine all models using simple average (equal weights)
grid_df["prob_ensemble"] = grid_df[["prob_glm", "prob_gam", "prob_dt", "prob_rf",  "prob_gb",  "prob_maxent"]].mean(axis=1)

model_probas_grid = np.stack([
    grid_df["prob_glm"],
    grid_df["prob_gam"],
    grid_df["prob_dt"],
    grid_df["prob_rf"],
    grid_df["prob_gb"],
    grid_df["prob_maxent"]
], axis=1)

#certainty of prediction from ensemble model
grid_df["ensemble_std"] = model_probas_grid.std(axis=1)
grid_df["certainty"]    = 1 - (grid_df["ensemble_std"] / grid_df["ensemble_std"].max())

print("Predictions done!\n")



Predictions done!



In [46]:
#build prediction map

#build hover text (show probability for each model + probability overall + lat & lon of point)
hover_text = [
    f"<b>VME Prediction</b><br>"
    f"----------------------<br>"
    f"GLM:              {glm:.1%}<br>"
    f"GAM:              {gam:.1%}<br>"
    f"Decision Tree:    {dt:.1%}<br>"
    f"Random Forest:    {rf:.1%}<br>"
    f"GradBoost:        {gb:.1%}<br>"
    f"MaxEnt:           {me:.1%}<br>"
    f"----------------------<br>"
    f"<b>Ensemble Score:  {ens:.1%}</b><br>"
    f"Certainty:        {cert:.1%}<br>"
    f"----------------------<br>"
    f"Lat: {lat:.2f},  Lon: {lon:.2f}<br>"
    f"Depth: {d:.0f} m"
    for glm, gam, dt, rf, gb, me, ens, cert, lat, lon, d in zip(
        grid_df["prob_glm"],
        grid_df["prob_gam"],
        grid_df["prob_dt"],
        grid_df["prob_rf"],
        grid_df["prob_gb"],
        grid_df["prob_maxent"],
        grid_df["prob_ensemble"],
        grid_df["certainty"],
        grid_df["Latitude"],
        grid_df["Longitude"],
        grid_df["depth"],
    )
]
 
fig = go.Figure()

# plot prediction points (color corresponds to overall probability)
fig.add_trace(go.Scattergeo(
    lon=grid_df["Longitude"],
    lat=grid_df["Latitude"],
    mode="markers",
    marker=dict(
        size=5,
        color=grid_df["prob_ensemble"],
        colorscale="RdYlBu_r",   # blue = low VME probability, red = high VME probability
        cmin=0,
        cmax=1,
        colorbar=dict(
            title="Ensemble VME<br>Likelihood",
            thickness=16,
            len=0.75,
            tickformat=".0%",
        ),
        opacity=0.85,
    ),
    text=hover_text,
    hoverinfo="text",
    name="VME Suitability",
))
 
fig.update_layout(
    title=dict(
        text=(
            "VME Probability — Indonesia Region<br>"
            "<sub>Colour corresponds to overall prediction probability of GLM, GAM, DT, RF, GB, MaxEnt models"
            " | Hover for per-model breakdown</sub>"
        ),
        font=dict(size=20),
        x=0.5,
    ),
    geo=dict(
        projection_type="mercator",
        lonaxis_range=[100, 145],
        lataxis_range=[-12, 7],
        showland=True,
        landcolor="rgb(200, 200, 200)",
        showocean=True,
        oceancolor="rgb(220, 235, 255)",
        showcoastlines=True,
        coastlinecolor="rgb(60, 60, 60)",
        showframe=False,
        bgcolor="white",
    ),
    height=700,
    width=1100,
    paper_bgcolor="white",
    margin=dict(t=100, b=20, l=20, r=20),
)

#save interactive map as html
output_path = "vme_prediction_map.html"
fig.write_html(output_path)

print(f"Saved -> {output_path}")
print("Open in any browser. Hover over dots to see VME probability at each point.")

Saved -> vme_prediction_map.html
Open in any browser. Hover over dots to see VME probability at each point.


## Results

The ensemble prediction map shows high possibility of VME in waters near North Kalimantan and Southwest Sulawesi. 